In [ ]:
import os

# Export these values in the kernel environment before running this notebook.
required_environment = ['SAGEMAKER_BUCKET', 'SAGEMAKER_EFS_ID', 'SAGEMAKER_IMAGE_URI', 'SAGEMAKER_ROLE_ARN', 'SAGEMAKER_SECURITY_GROUP_IDS', 'SAGEMAKER_SUBNET_IDS']
missing = [name for name in required_environment if not os.environ.get(name, "").strip()]
if missing:
    raise ValueError("Configure required environment variables: " + ", ".join(missing))


In [ ]:
from datetime import datetime, timezone
from sagemaker.core.helper.session_helper import Session
import boto3, os, tarfile, tempfile, json
from pathlib import Path

sagemaker_session = Session(default_bucket=os.environ["SAGEMAKER_BUCKET"])
region = sagemaker_session.boto_region_name
role = os.environ["SAGEMAKER_ROLE_ARN"]
bucket_name = os.environ["SAGEMAKER_BUCKET"]
sm = boto3.client('sagemaker', region_name=region)
s3 = boto3.client('s3', region_name=region)

instance_type = 'ml.r7i.4xlarge'
start_shard = 7
end_shard = 24
manifest_path = Path('../data/artifacts/seed_strategy/shards_24/launched_jobs_24.json')

print(region)
print(role)
print(instance_type, start_shard, end_shard)


In [ ]:
TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'

def tar_and_upload_source_code(source_path: str, bucket: str, s3_key_prefix: str='code'):
    source_path = os.path.abspath(source_path)
    with tempfile.TemporaryDirectory() as tmpdir:
        tar_path = os.path.join(tmpdir, TAR_SOURCE_FILENAME)
        with tarfile.open(tar_path, 'w:gz') as tar:
            tar.add(os.path.join(source_path, 'src'), arcname='src')
            tar.add(os.path.join(source_path, 'requirements-sagemaker.txt'), arcname='requirements-sagemaker.txt')
        s3_uri = f's3://{bucket}/{s3_key_prefix}/{TAR_SOURCE_FILENAME}'
        s3.upload_file(tar_path, bucket, f'{s3_key_prefix}/{TAR_SOURCE_FILENAME}')
        return s3_uri

source_code_uri = tar_and_upload_source_code('..', bucket_name, 'congestion-bogota/seed-time-artifact/code')
print(source_code_uri)


In [ ]:
num_shards = 24
time_idx_total = 2000
seed_strategy_prefix = 'congestion-bogota/seed-time-artifact/inputs/seeds'
time_idx_prefix = 'congestion-bogota/seed-time-artifact/inputs/time_idx'
seed_dir = Path('../data/artifacts/seed_strategy/shards_24')
time_idx_csv = Path('../data/artifacts/time_index_strategy/time_idx_strategy_2000.csv')

if not time_idx_csv.exists():
    raise ValueError(f'Missing time_idx strategy CSV: {time_idx_csv}')

for seed_csv in sorted(seed_dir.glob('seed_strategy_shard_*.csv')):
    s3.upload_file(str(seed_csv), bucket_name, f'{seed_strategy_prefix}/{seed_csv.name}')

s3.upload_file(str(time_idx_csv), bucket_name, f'{time_idx_prefix}/{time_idx_csv.name}')
print('uploaded shard seed CSVs and time_idx CSV')


In [ ]:
image = os.environ["SAGEMAKER_IMAGE_URI"]
NUMERIC_REGEX = r'([-+0-9eE\\.]+)'
metric_names = [
    'num_time_windows', 'train_time_indices', 'seed_candidates', 'node_batch_size',
    'input_seq_len', 'output_seq_len', 'pairs_evaluated', 'seeds_completed',
    'partial_score_mean', 'partial_score_min', 'partial_score_max', 'partial_low_ratio',
    'partial_medium_ratio', 'partial_high_ratio', 'num_seeds', 'num_time_indices',
    'score_min', 'score_mean', 'score_max', 'bucket_low_count', 'bucket_medium_count',
    'bucket_high_count',
]
metric_definitions_payload = [
    {'Name': name, 'Regex': rf'{name}={NUMERIC_REGEX}'} for name in metric_names
]


In [ ]:
existing_jobs = []
if manifest_path.exists():
    existing_jobs = json.loads(manifest_path.read_text())

launched_jobs = []
time_idx_s3 = f's3://{bucket_name}/{time_idx_prefix}/time_idx_strategy_2000.csv'

for shard_id in range(start_shard, end_shard):
    job_name = f'congestion-bogota-seed-time-shard-{shard_id:02d}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}'
    seed_s3 = f's3://{bucket_name}/{seed_strategy_prefix}/seed_strategy_shard_{shard_id:02d}.csv'
    request = {
        'TrainingJobName': job_name,
        'RoleArn': role,
        'AlgorithmSpecification': {
            'TrainingImage': image,
            'TrainingInputMode': 'File',
            'MetricDefinitions': metric_definitions_payload,
            'EnableSageMakerMetricsTimeSeries': True,
        },
        'HyperParameters': {
            'sagemaker_program': 'src/rebalance_sampling.py',
            'sagemaker_submit_directory': source_code_uri,
            'input-seq-len': '150',
            'output-seq-len': '50',
            'node-batch-size': '500',
            'train-ratio': '0.8',
            'include-weather': 'True',
            'include-velocity-target': 'True',
            'use-minimal-datasets': 'True',
            'high-level-threshold': '3.0',
            'bucket-quantiles': '0.5,0.75',
            'seed-candidates-path': seed_s3,
            'time-indices-path': time_idx_s3,
            'shard-id': str(shard_id),
            'num-shards': '1',
            'progress-log-every': '5000',
            'output-parquet': f'/opt/ml/model/seed_time_bucket_rows_shard_{shard_id:02d}.parquet',
            'output-metadata': f'/opt/ml/model/seed_time_bucket_metadata_shard_{shard_id:02d}.json',
            'output-parquet-s3-uri': f's3://{bucket_name}/congestion-bogota/seed-time-artifact/outputs/shard_{shard_id:02d}/seed_time_bucket_rows_shard_{shard_id:02d}.parquet',
            'output-metadata-s3-uri': f's3://{bucket_name}/congestion-bogota/seed-time-artifact/outputs/shard_{shard_id:02d}/seed_time_bucket_metadata_shard_{shard_id:02d}.json',
        },
        'InputDataConfig': [
            {
                'ChannelName': 'train',
                'DataSource': {
                    'FileSystemDataSource': {
                        'FileSystemId': os.environ["SAGEMAKER_EFS_ID"],
                        'FileSystemAccessMode': 'ro',
                        'FileSystemType': 'EFS',
                        'DirectoryPath': '/data/datos_procesados',
                    }
                },
            },
        ],
        'OutputDataConfig': {
            'S3OutputPath': f's3://{bucket_name}/congestion-bogota/seed-time-artifact/shards/{shard_id:02d}',
        },
        'ResourceConfig': {
            'InstanceType': instance_type,
            'InstanceCount': 1,
            'VolumeSizeInGB': 100,
        },
        'StoppingCondition': {'MaxRuntimeInSeconds': 7200},
        'VpcConfig': {
            'SecurityGroupIds': [v.strip() for v in os.environ["SAGEMAKER_SECURITY_GROUP_IDS"].split(",") if v.strip()],
            'Subnets': [v.strip() for v in os.environ["SAGEMAKER_SUBNET_IDS"].split(",") if v.strip()],
        },
    }
    sm.create_training_job(**request)
    launched_jobs.append({'shard_id': shard_id, 'job_name': job_name, 'instance_type': instance_type})
    manifest_path.write_text(json.dumps(existing_jobs + launched_jobs, indent=2))
    print(job_name, instance_type)

print('launched', len(launched_jobs), 'jobs')
